In [ ]:
#install database libraries
%pip install sqlalchemy psycopg2-binary

In [ ]:
#import libraries
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine, text

In [ ]:
#configure database connection
import getpass

db_user = "postgres"
db_password = getpass.getpass("PostgreSQL password:  ")
db_host = "localhost"
db_port = "5432"
db_name = "ai_data_analyst"

In [ ]:
#create database engine 
database_url = (
    f"postgresql+psycopg2://{db_user}:{db_password}"
    f"@{db_host}:{db_port}/{db_name}"
)

engine = create_engine(database_url)

In [ ]:
#testing database connection 
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT current_database(), version();")
    ).fetchone()

print("Connected database:", result[0])
print("PostgreSQL version:", result[1])

In [ ]:
#inspect database tables 
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
"""

with engine.connect() as connection:
    tables = pd.read_sql(text(query), connection)

tables

In [ ]:
#checking table row counts 
table_names = tables["table_name"].tolist()

with engine.connect() as connection:
    for table_name in table_names:
        result = connection.execute(
            text(f'SELECT COUNT(*) FROM "{table_name}"')
        ).scalar()

        print(f"{table_name}: {result:,} rows")

In [ ]:
#configuring csv files
csv_folder = Path("CSV")

print("CSV folder:", csv_folder.resolve())
print("CSV files:", len(list(csv_folder.glob("*.csv"))))

In [ ]:
#loading customers into PostgreSQL
customers = pd.read_csv(
    csv_folder / "olist_customers_dataset.csv"
)

customers.to_sql(
    "customers",
    engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000
)

print("Customers loaded successfully.")

In [ ]:
#verify customers table
with engine.connect() as connection:
    customer_count = connection.execute(
        text("SELECT COUNT(*) FROM customers")
    ).scalar()

print("PostgreSQL rows:", f"{customer_count:,}")
print("CSV rows:", f"{len(customers):,}")

In [ ]:
#loading category translation into PostgreSQL
category_translation = pd.read_csv(
    csv_folder / "product_category_name_translation.csv"
)

category_translation.to_sql(
    "category_translation",
    engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000
)

print("Category translation loaded successfully.")

In [ ]:
#loading products into PostgreSQL
products = pd.read_csv(
    csv_folder / "olist_products_dataset.csv"
)

products.to_sql(
    "products",
    engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000
)

print("Products loaded successfully.")

In [ ]:
#loading sellers
sellers = pd.read_csv(
    csv_folder / "olist_sellers_dataset.csv"
)

sellers.to_sql(
    "sellers",
    engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000
)

print("Sellers loaded successfully.")

In [ ]:
#verify 
expected_counts = {
    "category_translation": len(category_translation),
    "products": len(products),
    "sellers": len(sellers)
}

with engine.connect() as connection:
    for table_name, expected_count in expected_counts.items():
        database_count = connection.execute(
            text(f'SELECT COUNT(*) FROM "{table_name}"')
        ).scalar()

        print(
            f"{table_name}: "
            f"{database_count:,} PostgreSQL / "
            f"{expected_count:,} CSV"
        )

In [ ]:
#load orders
orders = pd.read_csv(
    csv_folder / "olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

orders.to_sql(
    "orders",
    engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000
)

print("Orders loaded successfully.")

In [ ]:
#verify order tables
with engine.connect() as connection:
    order_count = connection.execute(
        text("SELECT COUNT(*) FROM orders")
    ).scalar()

print("PostgreSQL rows:", f"{order_count:,}")
print("CSV rows:", f"{len(orders):,}")

In [ ]:
#load order items 
order_items = pd.read_csv(
    csv_folder / "olist_order_items_dataset.csv",
    parse_dates=["shipping_limit_date"]
)

order_items.to_sql(
    "order_items",
    engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000
)

print("Order items loaded successfully.")

In [ ]:
#verify
with engine.connect() as connection:
    order_items_count = connection.execute(
        text("SELECT COUNT(*) FROM order_items")
    ).scalar()

print("PostgreSQL rows:", f"{order_items_count:,}")
print("CSV rows:", f"{len(order_items):,}")

In [ ]:
#load payments
payments = pd.read_csv(
    csv_folder / "olist_order_payments_dataset.csv"
)

payments.to_sql(
    "payments",
    engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000
)

print("Payments loaded successfully.")

In [ ]:
#verify payments
with engine.connect() as connection:
    payments_count = connection.execute(
        text("SELECT COUNT(*) FROM payments")
    ).scalar()

print("PostgreSQL rows:", f"{payments_count:,}")
print("CSV rows:", f"{len(payments):,}")

In [ ]:
#load reviews
reviews = pd.read_csv(
    csv_folder / "olist_order_reviews_dataset.csv",
    parse_dates=[
        "review_creation_date",
        "review_answer_timestamp"
    ]
)

reviews.to_sql(
    "reviews",
    engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000
)

print("Reviews loaded successfully.")

In [ ]:
#verify reviews
with engine.connect() as connection:
    reviews_count = connection.execute(
        text("SELECT COUNT(*) FROM reviews")
    ).scalar()

print("PostgreSQL rows:", f"{reviews_count:,}")
print("CSV rows:", f"{len(reviews):,}")

In [ ]:
#load geolocation 
geolocation = pd.read_csv(
    csv_folder / "olist_geolocation_dataset.csv"
)

geolocation.to_sql(
    "geolocation",
    engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=10000
)

print("Geolocation loaded successfully.")

In [ ]:
#verify geolocation
with engine.connect() as connection:
    geolocation_count = connection.execute(
        text("SELECT COUNT(*) FROM geolocation")
    ).scalar()

print("PostgreSQL rows:", f"{geolocation_count:,}")
print("CSV rows:", f"{len(geolocation):,}")

In [ ]:
#verify complete database 
expected_counts = {
    "customers": 99441,
    "geolocation": 1000163,
    "order_items": 112650,
    "payments": 103886,
    "reviews": 99224,
    "orders": 99441,
    "products": 32951,
    "sellers": 3095,
    "category_translation": 71
}

with engine.connect() as connection:
    for table_name, expected_count in expected_counts.items():
        database_count = connection.execute(
            text(f'SELECT COUNT(*) FROM "{table_name}"')
        ).scalar()

        status = "OK" if database_count == expected_count else "MISMATCH"

        print(
            f"{table_name}: "
            f"{database_count:,} / {expected_count:,} - {status}"
        )

In [ ]:
#query total orders
